##### Написать программу, которая собирает входящие письма из своего или тестового почтового ящика и сложить данные о письмах в базу данных (от кого, дата отправки, тема письма, текст письма полный) Логин тестового ящика: study.ai_172@mail.ru Пароль тестового ящика: NextPassword172#

In [1]:
from functools import reduce
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.support.ui import WebDriverWait as WDW
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.by import By
from selenium.webdriver.common.action_chains import ActionChains
import time
import re
from pymongo import MongoClient

In [2]:
client = MongoClient('localhost',27017)
db = client['letters_db']
correspondence = db.correspondence

In [3]:
login = "study.ai_172"
pwd = "NextPassword172#"
url = "https://account.mail.ru/login/"

In [4]:
%%time
# Запуск и авторизация
chrome_options = Options()
chrome_options.add_argument("start-maximized")

s = Service('./chromedriver.exe')
driver = webdriver.Chrome(service=s, options=chrome_options)
driver.get(url)

login = WDW(driver, 30).until(EC.presence_of_element_located((By.NAME,'username')))
login.send_keys('study.ai_172')
login.submit()
password = WDW(driver, 30).until(EC.visibility_of_element_located((By.NAME,'password')))
password.send_keys('NextPassword172#')
password.submit()

#Подсчитаем количество писем в почтовом ящике
inbox = WDW(driver, 10).until(EC.visibility_of_element_located((By.CLASS_NAME,'nav__item_active')))
title = inbox.get_attribute('title')
m = [int(m) for m in str.split(title) if m.isdigit()]
mails = reduce(lambda x, y: x + y, m)
print(f"Писем в ящике: {mails}")

Писем в ящике: 499
Wall time: 7.42 s


In [5]:
%%time
#Собираем ссылки на письма
links_wait = WDW(driver, 30).until(EC.visibility_of_element_located((By.CLASS_NAME,'js-letter-list-item')))
links_list = driver.find_elements(By.CLASS_NAME,'js-letter-list-item')
links_box = set()

for a in links_list:
    links_box.add(a.get_attribute('href'))  

while len(links_box) != mails:
    actions = ActionChains(driver)
    actions.move_to_element(links_list[-1])
    actions.perform()
    time.sleep(1)
    links_list = driver.find_elements(By.CLASS_NAME,'js-letter-list-item')
    for a in links_list:
        links_box.add(a.get_attribute('href'))  
    print(f"Итого ссылок: {len(links_box)}")

Итого ссылок: 36
Итого ссылок: 47
Итого ссылок: 58
Итого ссылок: 69
Итого ссылок: 80
Итого ссылок: 91
Итого ссылок: 102
Итого ссылок: 113
Итого ссылок: 124
Итого ссылок: 135
Итого ссылок: 146
Итого ссылок: 157
Итого ссылок: 168
Итого ссылок: 179
Итого ссылок: 190
Итого ссылок: 201
Итого ссылок: 212
Итого ссылок: 223
Итого ссылок: 234
Итого ссылок: 245
Итого ссылок: 256
Итого ссылок: 267
Итого ссылок: 278
Итого ссылок: 289
Итого ссылок: 300
Итого ссылок: 311
Итого ссылок: 322
Итого ссылок: 333
Итого ссылок: 344
Итого ссылок: 355
Итого ссылок: 366
Итого ссылок: 377
Итого ссылок: 388
Итого ссылок: 399
Итого ссылок: 410
Итого ссылок: 421
Итого ссылок: 432
Итого ссылок: 443
Итого ссылок: 454
Итого ссылок: 465
Итого ссылок: 476
Итого ссылок: 487
Итого ссылок: 498
Итого ссылок: 499
Wall time: 1min 14s


In [6]:
%%time
#Собираем инфо из писем
content = []
for a in links_box:
    driver.get(a)
    letter_author_wrapper = WDW(driver, 100).until(EC.presence_of_element_located((By.CLASS_NAME, 'letter__author')))
    doc = {
        'author': letter_author_wrapper.find_element(By.CLASS_NAME,'letter-contact').get_attribute('title'),
        'date': letter_author_wrapper.find_element(By.CLASS_NAME,'letter__date').text,
        'title': driver.find_element(By.CLASS_NAME,'thread-subject').text,
        'body': driver.find_element(By.CLASS_NAME,'letter-body').text
    }
    content.append(doc)
    correspondence.update_one(doc, {'$set': doc}, upsert=True)
    print(f"Собрано из: {a}")

Собрано из: https://e.mail.ru/inbox/0:16446159781989525009:0/?authid=l082tlsb.51b&back=1%2C1&dwhsplit=s10273.b1ss12743a2s&from=login&x-login-auth=1&afterReload=1
Собрано из: https://e.mail.ru/inbox/0:16424359130553690351:0/?authid=l082tlsb.51b&back=1%2C1&dwhsplit=s10273.b1ss12743a2s&from=login&x-login-auth=1&afterReload=1
Собрано из: https://e.mail.ru/inbox/0:16452263461521295302:0/?authid=l082tlsb.51b&back=1%2C1&dwhsplit=s10273.b1ss12743a2s&from=login&x-login-auth=1&afterReload=1
Собрано из: https://e.mail.ru/inbox/0:16452197051467457390:0/?authid=l082tlsb.51b&back=1%2C1&dwhsplit=s10273.b1ss12743a2s&from=login&x-login-auth=1&afterReload=1
Собрано из: https://e.mail.ru/inbox/0:16448594431023205797:0/?authid=l082tlsb.51b&back=1%2C1&dwhsplit=s10273.b1ss12743a2s&from=login&x-login-auth=1&afterReload=1
Собрано из: https://e.mail.ru/inbox/0:16448451910443149561:0/?authid=l082tlsb.51b&back=1%2C1&dwhsplit=s10273.b1ss12743a2s&from=login&x-login-auth=1&afterReload=1
Собрано из: https://e.mail.r

In [7]:
content

[{'author': 'subscribe@garant.ru',
  'date': '12 февраля, 0:46',
  'title': 'ГАРАНТ. Рубрика "Социальная сфера" от 11.02.2022',
  'body': 'Письмо отображается некорректно? Посмотрите исходную версию на сайте!\n\n\nГАРАНТ. Рубрика "Социальная сфера" от 11.02.2022\n\nГорячие документы Информация Минфина\nи ФНС Бизнес-справки Бланки документов Форум\n\n  Присоединяйтесь к нам в:      \n  Новости\n11 февраля 2022\nВ Госдуму внесен законопроект об индексации военных пенсий на 8,6%\nПраво граждан с инвалидностью на бесплатное получение автомобиля могут восстановить\n10 февраля 2022\nПодготовлен обзор положений национальных стандартов ГОСТ Р 52877-2021, ГОСТ Р 53872-2021, ГОСТ Р 53873-2021, ГОСТ Р 54738-2021\n9 февраля 2022\nСкорректирован порядок реализации базовой программы ОМС в условиях распространения COVID-19\n7 февраля 2022\nМинздрав России разрешил оформлять больничные дистанционно при ОРВИ и COVID-19\nВыплаты гражданам, осуществляющим уход за инвалидами, могут повысить\nДобавить ГАРА

In [8]:
correspondence

Collection(Database(MongoClient(host=['localhost:27017'], document_class=dict, tz_aware=False, connect=True), 'letters_db'), 'correspondence')

In [9]:
for doc in correspondence.find():
    print(doc)

{'_id': ObjectId('621e0add9056aac0ed17e24d'), 'author': 'subscribe@garant.ru', 'body': 'Письмо отображается некорректно? Посмотрите исходную версию на сайте!\n\n\nГАРАНТ. Рубрика "Социальная сфера" от 11.02.2022\n\nГорячие документы Информация Минфина\nи ФНС Бизнес-справки Бланки документов Форум\n\n  Присоединяйтесь к нам в:      \n  Новости\n11 февраля 2022\nВ Госдуму внесен законопроект об индексации военных пенсий на 8,6%\nПраво граждан с инвалидностью на бесплатное получение автомобиля могут восстановить\n10 февраля 2022\nПодготовлен обзор положений национальных стандартов ГОСТ Р 52877-2021, ГОСТ Р 53872-2021, ГОСТ Р 53873-2021, ГОСТ Р 54738-2021\n9 февраля 2022\nСкорректирован порядок реализации базовой программы ОМС в условиях распространения COVID-19\n7 февраля 2022\nМинздрав России разрешил оформлять больничные дистанционно при ОРВИ и COVID-19\nВыплаты гражданам, осуществляющим уход за инвалидами, могут повысить\nДобавить ГАРАНТ.РУ в ваши источники\nАнонс\n25 февраля 2022 года